<a href="https://www.dask.org/" target="_blank">
<img src="http://dask.readthedocs.io/en/latest/_images/dask_horizontal.svg" align="right" width="30%" alt="Dask logo">
</a>

# Use case: Cloud-Native Satellite Image Processing (ARCO)

**Sentinel-2 NDVI straight from an Analysis-Ready, Cloud-Optimized store**

In the previous notebook (**Notebook 4**) we computed NDVI from a Sentinel-2
scene that had been **downloaded and stored on the cluster's shared NFS disk**
(`/home/data/raw`). That is the *traditional download-based* data-access model.

In this notebook we solve the **same problem** a different way: we **stream**
the data directly from an **Analysis-Ready Cloud-Optimized (ARCO)** Zarr store
in object storage, reading only the chunks we actually need. The goal is not
just to run new code, but to understand **why** the cloud-native approach scales
better than the download model when many students share one cluster — and what
it costs us in return.

**A. The download-based data-access model (recap of Notebook 4)**

In the **download model**, data is treated as something you must *bring to the
compute*. The workflow in Notebook 4 looked like this:

1. **Acquire** the full Sentinel-2 product (a `.SAFE` archive of `JP2` band
   files) from the data provider.
2. **Store** it once on the cluster's master node, `ss-00`, under
   `/home/data/raw`.
3. **Share** that directory over **NFS** so it is visible at the same path on
   all **13 compute nodes** (`ss-01` … `ss-13`).
4. **Access** it from the notebook with `rioxarray.open_rasterio(...)`, pointing
   at full paths inside the SAFE archive, e.g.:

   ```python
   images_storage_raw = '/home/data/raw'
   red_file = f'{images_storage_raw}/{image_id}/.../R10m/..._B04_10m.jp2'
   ```

Every band read by every Dask worker — on any of the 13 nodes — travels back
through the **single NFS server on `ss-00`**. That central dependency is the key
to understanding the model's limits.

**B. Download-based vs. cloud-native: a side-by-side view**

The two workflows differ mainly in **where the data lives** and **how it reaches
the workers**. In the download model everything funnels through one NFS server;
in the cloud-native model each worker pulls only the chunks it needs, in
parallel, directly from object storage.

<img src="img/download_vs_cloud.svg" width="600px">

*If the diagram does not render in your viewer, the key idea is: **(A)** all 13
nodes read through one NFS server on `ss-00`; **(B)** all 13 workers read their
own chunks directly from object storage, with no shared server in the middle.*

**C. Advantages and limitations of the download model**

**Where the download model is reasonable.** It is simple and familiar: the data
sits on a normal POSIX path, any tool (`gdal`, `rasterio`, even plain file I/O)
can open it, it works fully offline once downloaded, and reads from a warm NFS
cache can be fast for a *single* user. For a one-off analysis on a small team,
it is perfectly adequate.

**Where it breaks down.** The limitations below are about **data movement and
shared-resource contention**, not about a single machine running out of RAM:

1. **Heavy reliance on the shared NFS server.** All band reads from all 13 nodes
   funnel through the one NFS daemon on `ss-00`. That server's network link and
   disk become the ceiling for the whole cluster.
2. **Poor scalability under concurrent access.** With ~20 students each launching
   Dask workers across the nodes, dozens of processes hammer the *same* NFS
   export at once. Throughput per user drops and latency rises as concurrency
   grows — the workflow gets **slower precisely when more people use it**.
3. **Storage formats not optimized for chunked access.** The Sentinel-2 bands are
   `JP2` files inside a `.SAFE` archive. To read a small spatial window you still
   pay for opening/decoding overhead, and the format was not designed for
   independent, parallel chunk reads the way Zarr/COG are.
4. **You must download and maintain local copies.** Someone has to fetch the
   scene, unpack the archive, place it on `/home/data/raw`, and keep it there —
   plus manage versions, cleanup, and disk quotas over time.
5. **Repeated downloads of the same data.** Without coordination, the same scene
   is fetched again and again (per user, per project, per re-run), wasting
   bandwidth and time.
6. **Increased data movement and storage footprint.** A full scene includes many
   bands and resolutions you may never use, yet the *entire* product is copied
   and stored. You pay in transfer time and disk for data you do not touch.

**D. Advantages and trade-offs of the ARCO / cloud-native approach**

*Advantages.*

- **Chunked, selective access.** Data is stored as many small, independently
  addressable chunks. You read *only* the bands, area, and time steps you need —
  not the whole scene.
- **Streaming, not downloading.** Chunks are pulled on demand at compute time.
  There is no separate "download and unpack" step and no local copy to manage.
- **Scales for distributed computing.** Each Dask worker fetches its own chunks
  directly from object storage in parallel. Object stores are built for many
  simultaneous range requests, so adding workers (or students) adds throughput
  instead of fighting over one file server.
- **Reduced data movement.** Only the chunks touched by the computation cross the
  network; unused bands/resolutions are never transferred.
- **Native Dask + Xarray integration.** `xarray.open_zarr` returns lazy,
  Dask-backed arrays with labelled dimensions, so the analysis code is essentially
  the same as for local data — but distributed and lazy by default.

*Trade-offs and disadvantages.*

- **Dependence on network connectivity.** No network (or a slow link to the
  object store) means no data. The offline simplicity of a local copy is lost.
- **Higher latency for some access patterns.** Each chunk read is a network
  request. Workloads that touch many tiny, scattered chunks can suffer from
  per-request latency compared with a warm local cache.
- **More moving parts.** Cloud-native formats and access (Zarr layout, chunking
  strategy, `s3fs`/endpoints, consolidated metadata, compression codecs) add
  conceptual and operational complexity over "it's just a file."
- **Potential storage and egress costs.** On commercial clouds, object storage
  and especially **egress** (data leaving the provider) can cost money. Chunk
  size and access patterns directly affect that bill. *(The store used here lives
  on the public, no-cost OSN pod, so we do not pay egress — but the concern is
  real in production.)*

**What is ARCO?**

**ARCO — Analysis-Ready, Cloud-Optimized — data** is data that has been prepared
so it can be analyzed *directly from object storage*, with little or no
pre-processing, by many consumers at once. Two ideas combine:

- **Analysis-Ready:** already cleaned and standardized — consistent grid,
  projection, units, and metadata — so you can compute on it immediately instead
  of wrangling raw products.
- **Cloud-Optimized:** stored in a format whose internal layout supports
  **partial, parallel reads over HTTP** — small chunks, internal indexing, and
  compression. Common examples are **Zarr** (used here) and **Cloud-Optimized
  GeoTIFF (COG)**.

**Why it matters for large-scale geospatial analytics.**

- **Efficient subsetting.** Because the data is chunked and indexed, a query for
  one region, a few bands, or specific dates reads just those chunks — turning a
  multi-gigabyte scene into a few megabytes of actual transfer.
- **Parallel-friendly.** Independent chunks map naturally onto Dask tasks, so the
  same store feeds dozens of workers without a central bottleneck.
- **Shareable and reproducible.** One canonical, versioned store is read by
  everyone — no per-user downloads, no divergent local copies.

<img src="img/arco.webp" width="600px">

Reference: [arco-the-smartest-way-to-access-big-geospatial-data](https://blog.lobelia.earth/arco-the-smartest-way-to-access-big-geospatial-data-eaf689eff3c9)

In this notebook, the ARCO store is a **Zarr** dataset of the same Sentinel-2 tile
used in Notebook 4, pre-built once from the raw SAFE scenes by
`source/build_arco.py` and published to a **public OSN bucket**:

```text
s3://colombia-radar-arco/sentinel2-ard/T18NYM_20200205_20200210.zarr
endpoint: https://umn1.osn.mghpcc.org   (public, anonymous read)
```

It holds the native 10 m bands (`b02, b03, b04, b08`) stacked along a `band`
dimension, a true-color `tci` variable, and **two acquisitions stacked along
`time`** — all chunked for streaming access.

> **New to xarray or Zarr?** Two good primers:
> [Introducción a xarray](https://aladinor.github.io/AtmosCol-2023/introduccion-xarray/)
> (labelled N-D arrays, lazy/Dask-backed reads) and
> [*From Tensors to Clouds: A Practical Guide to Zarr V3*](https://www.youtube.com/watch?v=8FX4AXhRNMA)
> (chunked, compressed, cloud-native array storage).


## 1. Hands-on: NDVI from the ARCO store

We now reproduce the NDVI result from Notebook 4 — but reading from the chunked
Zarr store instead of NFS. The flow is: start a Dask cluster, open the store
lazily, compute NDVI per chunk, trigger the distributed computation, and plot.

**Section objective:** open a remote Zarr store with Xarray, build a *lazy* NDVI
computation, and let Dask stream and process the chunks across the cluster.

### 1.1 Import the libraries

We use `s3fs` to talk to the object store, `xarray` for the labelled, lazy data
model, `matplotlib` for the colormap, and Dask's `SLURMCluster` + `Client` to run
the work across the cluster's compute nodes.

In [ ]:
import s3fs
import xarray as xr
import matplotlib as mpl
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

### 1.2 Start a Dask cluster

We create the cluster **first** so every operation below runs on the distributed
scheduler — open the **Dashboard** link in the cell output to watch chunks stream
from the object store and NDVI tasks execute live.

Here we use the same `SLURMCluster` as Notebook 4 (each worker is a Slurm job on
a compute node) and scale to 2 worker jobs. On a laptop you could instead swap in
a `dask.distributed.LocalCluster` with the same downstream code.

In [ ]:
cluster = SLURMCluster(
    cores=1, 
    memory="8GB",
    processes=True,
    scheduler_options={"dashboard_address": ":0"}
)

cluster.scale(jobs=2)

client = Client(cluster)
client

### 1.3 Open the ARCO store from OSN (anonymous)

The `colombia-radar-arco` bucket is **public-read**, so we open it with
`anon=True` — no credentials required. `xarray.open_zarr` returns **lazy,
Dask-backed** arrays: nothing is transferred yet. The arrays keep the store's
native chunking (here `100 x 100`), which is what makes the per-chunk parallel
reads possible.

In [ ]:
fs = s3fs.S3FileSystem(
    anon=True,
    client_kwargs={"endpoint_url": "https://umn1.osn.mghpcc.org"},
)
store = s3fs.S3Map(
    "colombia-radar-arco/sentinel2-ard/T18NYM_20200205_20200210.zarr",
    s3=fs,
    check=False,
)

ds = xr.open_zarr(store, consolidated=False)  # lazy, dask-backed
ds

Inspect the repr above: the dataset stacks the native 10 m bands
(`b02, b03, b04, b08`, raw `uint16` digital numbers) along `band`, includes a
true-color `tci` variable, and holds **two acquisitions along `time`**. Notice
the variables are **Dask arrays** — still nothing downloaded.

### 5.4 (Optional) Explore individual bands

The same per-band views as Notebook 4, but served straight from the ARCO store —
no unzip, no full-tile load. Each plot triggers reads of **only** the chunks it
needs. `isel(time=0)` selects the first acquisition (2020-02-05). **Uncomment**
to run.

In [ ]:
# True-color composite (TCI)
# ds.tci.isel(time=0).plot.imshow(figsize=(8, 7))

# Red band (B04)
# ds.reflectance.sel(band="b04").isel(time=0).plot(figsize=(8, 7))

# Near-infrared band (B08)
# ds.reflectance.sel(band="b08").isel(time=0).plot(figsize=(8, 7))

### 1.5 Define the NDVI computation (lazy)

NDVI = (NIR − Red) / (NIR + Red) = (`b08` − `b04`) / (`b08` + `b04`). We build the
expression on the **chunked** arrays, so it stays lazy — Dask records the task
graph but does not execute yet. Because NDVI is a *ratio*, it is invariant to the
digital-number-to-reflectance scale, so the result matches Notebook 4 exactly. We
mask `0/0` nodata pixels with `.where(denom > 0)`.

In [ ]:
refl = ds.reflectance.astype("float32")
nir = refl.sel(band="b08")
red = refl.sel(band="b04")

denom = nir + red
ndvi = ((nir - red) / denom).where(denom > 0)  # mask nodata (0/0)
ndvi

### 1.6 Trigger the distributed computation

Calling `.compute()` submits the per-chunk tasks to the cluster — watch the
**Dashboard** light up as workers stream their chunks from OSN and compute NDVI
in parallel. Only the (small) result for the first acquisition is pulled back
into memory here.

In [ ]:
result = ndvi.isel(time=0).compute()  # distributed over the 100x100 chunks
result

### 5.7 Plot NDVI

We reuse the same classified colormap as Notebook 4 so the two notebooks are
directly comparable. Values near **1** are dense, healthy vegetation; near **0**
sparse vegetation; negative values typically water.

In [ ]:
cmap = mpl.colors.ListedColormap(
    [
        "#000000", "#a50026", "#d73027", "#f46d43",
        "#fdae61", "#fee08b", "#ffffbf", "#d9ef8b",
        "#a6d96a", "#66bd63", "#1a9850", "#006837",
    ]
)
bounds = [-1.0, -0.2, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
norm = mpl.colors.BoundaryNorm(bounds, cmap.N)

result.plot(cmap=cmap, norm=norm, figsize=(8, 7))

### 1.8 Bonus: NDVI change between the two dates

Because both scenes are time-stacked in **one** store on the same grid, we can
difference them directly — something the single-scene download notebook could not
do without fetching and aligning a second product. This also runs on the cluster.

> **Caveat:** these two acquisitions are only **5 days apart** (2020-02-05 and
> 2020-02-10). Vegetation changes little over such a short interval, so this
> difference mainly highlights **clouds, water-level changes, and
> illumination/atmospheric differences** rather than real vegetation dynamics. It
> demonstrates the *mechanics* of a time-series difference on an ARCO cube; for
> meaningful phenological change you would stack scenes weeks-to-months apart.

In [ ]:
ndvi_diff = (ndvi.isel(time=1) - ndvi.isel(time=0)).compute()
ndvi_diff.plot(cmap="RdBu", vmin=-0.5, vmax=0.5, figsize=(8, 7))

### 1.9 Release the cluster resources

Always shut down the client and cluster when finished so the Slurm worker jobs
are returned to the pool for other students.

In [ ]:
client.close()
cluster.close()

## Summary & key takeaways

- The **download model** (Notebook 4) brings whole scenes to the compute and
  shares them over **NFS**. Its real limits are **shared-server I/O and poor
  scalability under concurrent access** — *not* memory.
- The **cloud-native / ARCO model** (this notebook) leaves data in object storage
  and **streams only the chunks needed**, so many workers — and many students —
  read in parallel without a central bottleneck.
- ARCO is not free of trade-offs: it depends on the **network**, can add
  **per-request latency** and **format/infra complexity**, and on commercial
  clouds can incur **storage/egress costs**.
- **ARCO = Analysis-Ready + Cloud-Optimized:** standardized, chunked, indexed data
  (e.g. **Zarr**, **COG**) that enables efficient subsetting and parallel
  processing — a natural fit for **Dask + Xarray** at scale.